# US Structure Data Pipeline

Build city-level structure polygons for U.S. cities by combining:

- **Overture Maps buildings** for primary polygons and building attributes.
- **Microsoft Global ML Building Footprints** as a polygon gap-filler and height/confidence source.
- **OSMnx / OpenStreetMap** for building tags, names, floors, units, and heights where mapped.
- **USACE National Structure Inventory (NSI)** for occupancy type, stories, residential units, population proxies, employees, students, values, and Census block IDs.
- **Census ACS** for a city-level average household-size fallback when structure-level occupant counts are not available.
- **Optional parcel files or public ArcGIS parcel layers** for parcel ID, address, land use, zoning, owner, value, and year-built context.

The output is one row per structure polygon with raw source fields collapsed into auditable columns such as `StructureTypeSource`, `NumUnitsSource`, `NumStoriesSource`, `OccupantCountMethod`, and `ParcelMatchMethod`.

## Install Dependencies

Run this once if the environment is missing packages. These are ordinary Python/geospatial packages; no AI model packages are used.

```python
%pip install -r requirements.txt
```

In [1]:
import importlib
import structure_pipeline

importlib.reload(structure_pipeline)
from structure_pipeline import PipelineConfig, build_many_cities

## Configure Cities and Sources

Add as many cities as needed. For large cities, the first run can take time because OSM, Overture, Microsoft, NSI, Census, and optional parcel calls are network-bound. Local source files are reused on later runs.

Parcel data is local-government specific. Add a source only when you have a parcel file or public ArcGIS FeatureServer layer for that city.

In [2]:
CITIES = [
    {"city": "Chicago", "state": "Illinois"},
    # Example with a local parcel file:
    # {
    #     "city": "Houston",
    #     "state": "Texas",
    #     "parcel_source": {
    #         "path": "data/raw/houston_parcels.gpkg",
    #         "field_map": {
    #             "ParcelID": "ACCOUNT",
    #             "ParcelAddress": "SITUS_ADDR",
    #             "ParcelLandUse": "LAND_USE",
    #             "ParcelZoning": "ZONING",
    #             "ParcelOwner": "OWNER_NAME",
    #             "ParcelAssessedValue": "TOTAL_VALUE",
    #             "ParcelYearBuilt": "YEAR_BUILT",
    #         },
    #     },
    # },
    # {"city": "Austin", "state": "Texas"},
    # {"city": "Seattle", "state": "Washington"},
]

config = PipelineConfig(
    data_dir="data",
    output_dir="data/output",
    raw_dir="data/raw",
    cache_dir="cache",
    country="USA",
    download_missing=True,
    use_overture=True,
    use_microsoft=True,
    use_osm=True,
    use_nsi=True,
    use_census=True,
    use_parcels=True,
    add_microsoft_unmatched=True,
    # Alternative: configure parcel sources by city slug.
    parcel_sources={
        # "houston_texas_usa": {
        #     "url": "https://.../FeatureServer/0",
        #     "field_map": {"ParcelID": "PARCEL_ID"},
        # },
    },
    # Smaller values reduce NSI response memory per request but increase request count.
    nsi_tile_size_deg=0.08,
)

## Run Pipeline

Outputs are written to `data/output/{city}_{state}_usa_structures.parquet` and a combined `data/output/structures_master.parquet`.

In [3]:
structures = build_many_cities(CITIES, config)
structures.head()


=== Chicago, Illinois ===
Overture polygons: 832,144
Microsoft polygons: 673,538
Merged footprint polygons: 844,132
OSM source skipped: HTTPSConnectionPool(host='overpass-api.de', port=443): Read timed out. (read timeout=180)
OSM building polygons: 0
NSI tile skipped: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
NSI structure points: 562,721
Parcels skipped: no parcel source configured for this city
Parcel polygons: 0
Saved 844,132 rows to data/output/chicago_illinois_usa_structures.parquet
Saved combined output to data/output/structures_master.parquet


,StructureID,City,State,Country,FootprintSource,OvertureID,MicrosoftID,OSMID,NSI_FD_ID,NSI_RecordCount,...,ParcelYearBuilt,ParcelArea_m2,ParcelSource,ParcelMatchMethod,FootprintArea_m2,Confidence_MS,HasParts,CensusAvgHouseholdSize,CensusSource,geometry
0,ovt_11fecb2b-8d33-4d78-8630-c2f35994c466,Chicago,Illinois,USA,overture,11fecb2b-8d33-4d78-8630-c2f35994c466,<NA>,<NA>,<NA>,NaN,...,NaN,NaN,<NA>,<NA>,58.314727,NaN,False,2.29,ACS 2024 B25010_001E,"POLYGON ((-87.63394 41.6578, -87.63394 41.6577..."
1,ovt_9e9d7e2c-d424-49b9-8b7c-dc681f152c04,Chicago,Illinois,USA,overture,9e9d7e2c-d424-49b9-8b7c-dc681f152c04,<NA>,<NA>,<NA>,NaN,...,NaN,NaN,<NA>,<NA>,65.815284,NaN,False,2.29,ACS 2024 B25010_001E,"POLYGON ((-87.63377 41.65782, -87.63376 41.657..."
2,ovt_b85243ae-1115-4101-a58d-6d43d0c13fd1,Chicago,Illinois,USA,overture,b85243ae-1115-4101-a58d-6d43d0c13fd1,<NA>,<NA>,<NA>,NaN,...,NaN,NaN,<NA>,<NA>,48.309941,NaN,False,2.29,ACS 2024 B25010_001E,"POLYGON ((-87.63369 41.65782, -87.63368 41.657..."
3,ovt_3206d556-ee08-4887-a816-b7bb70ac221e,Chicago,Illinois,USA,overture,3206d556-ee08-4887-a816-b7bb70ac221e,<NA>,<NA>,<NA>,NaN,...,NaN,NaN,<NA>,<NA>,32.021605,NaN,False,2.29,ACS 2024 B25010_001E,"POLYGON ((-87.63602 41.65767, -87.63602 41.657..."
4,ovt_a69ccb2e-33ff-4892-9a22-ca2ec829c127,Chicago,Illinois,USA,overture,a69ccb2e-33ff-4892-9a22-ca2ec829c127,<NA>,<NA>,<NA>,NaN,...,NaN,NaN,<NA>,<NA>,84.823924,NaN,False,2.29,ACS 2024 B25010_001E,"POLYGON ((-87.63621 41.65773, -87.63621 41.657..."


## Inspect Coverage

These checks show how often the final fields came from each source or inference method.

In [4]:
print("Rows:", len(structures))
print("Cities:", structures[["City", "State"]].drop_duplicates().to_dict("records"))

summary_cols = [
    "FootprintSource",
    "StructureTypeSource",
    "NumUnitsSource",
    "NumStoriesSource",
    "HeightSource",
    "OccupantCountMethod",
    "ParcelSource",
    "ParcelMatchMethod",
]
for col in summary_cols:
    print(f"\n{col}")
    print(structures[col].value_counts(dropna=False).head(20))

Rows: 844132
Cities: [{'City': 'Chicago', 'State': 'Illinois'}]

FootprintSource
FootprintSource
overture     832144
microsoft     11988
Name: count, dtype: int64

StructureTypeSource
StructureTypeSource
nsi_occtype         470675
<NA>                327374
overture_class       45290
overture_subtype       793
Name: count, dtype: int64

NumUnitsSource
NumUnitsSource
<NA>                      562995
inferred_single_family    281137
Name: count, dtype: int64

NumStoriesSource
NumStoriesSource
overture           424590
height_estimate    222295
nsi                126758
<NA>                70489
Name: count, dtype: int64

HeightSource
HeightSource
overture     749409
<NA>          84483
microsoft     10240
Name: count, dtype: int64

OccupantCountMethod
OccupantCountMethod
nsi_max_2am_2pm_emp_students         497219
NaN                                  344930
num_units_x_census_household_size      1983
Name: count, dtype: int64

ParcelSource
ParcelSource
<NA>    844132
Name: count, dtype: 

## Output Schema

Key normalized columns:

- `geometry`: structure polygon in EPSG:4326.
- `StructureType`: normalized type such as `residential`, `condo`, `apartment`, `commercial`, `hotel`, `garage`, `barn`, `industrial`, `warehouse`, `education`, `healthcare`, or `unknown`.
- `StructureTypeRaw` and `StructureTypeSource`: raw value and source used to derive `StructureType`.
- `NumUnits` and `NumUnitsSource`: explicit OSM/NSI residential units, or `inferred_single_family` when the source type clearly describes a single-family structure.
- `NumStories` and `NumStoriesSource`: OSM floors, Overture floors, NSI stories, or a height-derived estimate.
- `OccupantCount` and `OccupantCountMethod`: NSI population/employee/student proxy when present, otherwise `NumUnits * CensusAvgHouseholdSize` for residential structures.
- `CBFIPS`: Census block FIPS from NSI where available.
- `ParcelID`, `ParcelAddress`, `ParcelLandUse`, `ParcelZoning`, `ParcelOwner`, `ParcelAssessedValue`, `ParcelYearBuilt`, and `ParcelArea_m2`: parcel context when a parcel source is configured.
- `ParcelMatchMethod`: currently `representative_point_within`, meaning the structure's representative point fell inside the parcel polygon.

Occupant counts should be treated as estimates unless your downstream workflow has a stronger authoritative source.

In [5]:
columns = [
    "StructureID", "City", "State", "FootprintSource", "StructureType",
    "StructureTypeRaw", "StructureTypeSource", "NumUnits", "NumUnitsSource",
    "NumStories", "NumStoriesSource", "OccupantCount", "OccupantCountMethod",
    "CBFIPS", "ParcelID", "ParcelAddress", "ParcelLandUse", "ParcelZoning",
    "ParcelAssessedValue", "ParcelYearBuilt", "ParcelArea_m2",
    "ParcelMatchMethod", "FootprintArea_m2", "geometry",
]
structures[columns].head(100)

,StructureID,City,State,FootprintSource,StructureType,StructureTypeRaw,StructureTypeSource,NumUnits,NumUnitsSource,NumStories,...,ParcelID,ParcelAddress,ParcelLandUse,ParcelZoning,ParcelAssessedValue,ParcelYearBuilt,ParcelArea_m2,ParcelMatchMethod,FootprintArea_m2,geometry
0,ovt_11fecb2b-8d33-4d78-8630-c2f35994c466,Chicago,Illinois,overture,unknown,<NA>,<NA>,NaN,<NA>,2.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,58.314727,"POLYGON ((-87.63394 41.6578, -87.63394 41.6577..."
1,ovt_9e9d7e2c-d424-49b9-8b7c-dc681f152c04,Chicago,Illinois,overture,unknown,<NA>,<NA>,NaN,<NA>,4.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,65.815284,"POLYGON ((-87.63377 41.65782, -87.63376 41.657..."
2,ovt_b85243ae-1115-4101-a58d-6d43d0c13fd1,Chicago,Illinois,overture,unknown,<NA>,<NA>,NaN,<NA>,1.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,48.309941,"POLYGON ((-87.63369 41.65782, -87.63368 41.657..."
3,ovt_3206d556-ee08-4887-a816-b7bb70ac221e,Chicago,Illinois,overture,unknown,<NA>,<NA>,NaN,<NA>,1.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,32.021605,"POLYGON ((-87.63602 41.65767, -87.63602 41.657..."
4,ovt_a69ccb2e-33ff-4892-9a22-ca2ec829c127,Chicago,Illinois,overture,unknown,<NA>,<NA>,NaN,<NA>,1.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,84.823924,"POLYGON ((-87.63621 41.65773, -87.63621 41.657..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,ovt_ba6bc2d3-0ae9-467f-aaf4-8c7b595086ab,Chicago,Illinois,overture,residential,RES1-1SWB,nsi_occtype,1.0,inferred_single_family,1.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,92.455456,"POLYGON ((-87.63982 41.65874, -87.63981 41.658..."
96,ovt_fb440a9c-6888-46f7-8f00-5d49c52a79c6,Chicago,Illinois,overture,residential,RES1-1SWB,nsi_occtype,1.0,inferred_single_family,1.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,121.104266,"POLYGON ((-87.63941 41.65877, -87.63941 41.658..."
97,ovt_8300c520-caf2-49b0-a592-83470e195a0f,Chicago,Illinois,overture,unknown,<NA>,<NA>,NaN,<NA>,1.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,58.431846,"POLYGON ((-87.63986 41.65882, -87.63986 41.658..."
98,ovt_33f6f49b-fe4a-4001-9dde-ce69c9cc284c,Chicago,Illinois,overture,unknown,<NA>,<NA>,NaN,<NA>,1.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,28.200745,"POLYGON ((-87.63931 41.65887, -87.6393 41.6588..."
